In [1]:
from pathlib import Path
import pandas as pd
DATA = Path("data")
talks = pd.read_csv(DATA / "demo_talks.csv")
speakers = pd.read_csv(DATA / "demo_speakers.csv")
speakers_conflicting = pd.read_csv(DATA / "demo_speakers_conflicting.csv")
def show(label, value):
    print("\n" + label)
    print(value.to_string(index=False) if isinstance(value, pd.DataFrame) else value)
show("talks", talks)
show("speakers", speakers)



talks
talk_id speaker_id  num_words
     T1         S1       1000
     T2         S1       1200
     T3         S1       1400
     T4         S2       2400
     T5         S3       1600

speakers
speaker_id speaker group
        S1   Avery     A
        S2   Blair     B
        S4   Casey     B


In [2]:
left = pd.merge(talks, speakers, on="speaker_id", how="left")
show("left", left)



left
talk_id speaker_id  num_words speaker group
     T1         S1       1000   Avery     A
     T2         S1       1200   Avery     A
     T3         S1       1400   Avery     A
     T4         S2       2400   Blair     B
     T5         S3       1600     NaN   NaN


In [3]:
inner = pd.merge(talks, speakers, on="speaker_id", how="inner")
show("inner", inner)



inner
talk_id speaker_id  num_words speaker group
     T1         S1       1000   Avery     A
     T2         S1       1200   Avery     A
     T3         S1       1400   Avery     A
     T4         S2       2400   Blair     B


In [4]:
audited = pd.merge(talks, speakers, on="speaker_id", how="left", indicator=True)
show("Matching status", audited[["talk_id", "speaker_id", "group", "_merge"]])
# both: a matching key in each input.
# left_only: a retained left row without a matching right key.
# right_only: a retained right row without a matching left key.
# A left join has no right_only rows. Outer/right joins can retain them.



Matching status
talk_id speaker_id group    _merge
     T1         S1     A      both
     T2         S1     A      both
     T3         S1     A      both
     T4         S2     B      both
     T5         S3   NaN left_only


In [5]:
mask = audited["_merge"].eq("left_only")
# Equivalent familiar syntax: mask = audited["_merge"] == "left_only"
show("Elementwise comparison", pd.DataFrame({
    "talk_id": audited["talk_id"], "_merge": audited["_merge"], "mask": mask,
}))
unmatched = audited.loc[mask]
show("unmatched", unmatched[["talk_id", "speaker_id", "_merge"]])



Elementwise comparison
talk_id    _merge  mask
     T1      both False
     T2      both False
     T3      both False
     T4      both False
     T5 left_only  True

unmatched
talk_id speaker_id    _merge
     T5         S3 left_only


In [6]:
summary = (
    left.groupby("group", dropna=False, as_index=False)
    .agg(n_talks=("talk_id", "size"), mean_words=("num_words", "mean"))
)
show("summary", summary)



summary
group  n_talks  mean_words
    A        3      1200.0
    B        1      2400.0
  NaN        1      1600.0


In [7]:
bad = pd.merge(talks, speakers_conflicting, on="speaker_id", how="left")
show("Complete flawed join", bad[["talk_id", "speaker_id", "num_words", "group"]])



Complete flawed join
talk_id speaker_id  num_words group
     T1         S1       1000     A
     T1         S1       1000     B
     T2         S1       1200     A
     T2         S1       1200     B
     T3         S1       1400     A
     T3         S1       1400     B
     T4         S2       2400     B
     T5         S3       1600   NaN


In [8]:
activity_talks = pd.read_csv(DATA / "activity_talks.csv")
speakers_conflicting = pd.read_csv(DATA / "activity_speakers_conflicting.csv")
try:
    pd.merge(activity_talks, speakers_conflicting, on="speaker_id", how="left", validate="many_to_one")
except pd.errors.MergeError as error:
    print("MergeError:", error)


MergeError: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
 speaker_id
        S1 ...


In [9]:
current = speakers_conflicting.loc[speakers_conflicting["record_status"].eq("current")]
fixed = pd.merge(activity_talks, current, on="speaker_id", how="left", validate="many_to_one", indicator=True)
show("fixed", fixed[["talk_id", "group", "_merge"]])



fixed
talk_id group    _merge
    A01     A      both
    A02     A      both
    A03     A      both
    A04     A      both
    A05     B      both
    A06     B      both
    A07     B      both
    A08     A      both
    A09     A      both
    A10     B      both
    A11     B      both
    A12   NaN left_only


In [10]:
comparison = pd.DataFrame({
    "quantity": ["rows", "total_words", "mean_words"],
    "original": [len(talks), talks["num_words"].sum(), talks["num_words"].mean()],
    "flawed_join": [len(bad), bad["num_words"].sum(), bad["num_words"].mean()],
})
show("comparison", comparison)



comparison
   quantity  original  flawed_join
       rows       5.0          8.0
total_words    7600.0      11200.0
 mean_words    1520.0       1400.0


In [11]:
conference_metadata = pd.read_csv(DATA / "conference_metadata.csv")
with_location = pd.merge(activity_talks, conference_metadata, on=["year", "season"], how="left", validate="many_to_one")
show("with_location", with_location[["talk_id", "year", "season", "location"]])



with_location
talk_id  year  season   location
    A01  2025   April North Hall
    A02  2025   April North Hall
    A03  2025 October South Hall
    A04  2025 October South Hall
    A05  2025   April North Hall
    A06  2025 October South Hall
    A07  2026   April  East Hall
    A08  2026   April  East Hall
    A09  2026 October  West Hall
    A10  2026   April  East Hall
    A11  2026 October  West Hall
    A12  2026 October  West Hall


In [12]:
show("Unique talk IDs", talks["talk_id"].is_unique)
show("Unique speaker IDs", speakers["speaker_id"].is_unique)
show("Missing speaker keys", speakers["speaker_id"].isna().sum())



Unique talk IDs
True

Unique speaker IDs
True

Missing speaker keys
0


In [13]:
directory = speakers.rename(columns={"speaker_id": "person_id"})
renamed = pd.merge(talks, directory, left_on="speaker_id", right_on="person_id", how="left", validate="many_to_one")
show("renamed", renamed[["talk_id", "person_id", "speaker"]])



renamed
talk_id person_id speaker
     T1        S1   Avery
     T2        S1   Avery
     T3        S1   Avery
     T4        S2   Blair
     T5       NaN     NaN


In [ ]:
left_keys = pd.DataFrame({"key": [None], "x": [1]})
right_keys = pd.DataFrame({"key": [None], "y": [2]})
show("Missing key match", pd.merge(left_keys, right_keys, on="key", how="inner"))



Missing keys match
 key  x  y
None  1  2
